In [1]:
import pandas as pd

df = pd.read_csv('../data/movies_dataset.csv')

df = df.drop(columns=['Unnamed: 0'])

df.head()

df['User_Id'].nunique()

df['Movie_Name'].nunique()

df['Genre'].nunique()

df['Movie_Name'].value_counts().head(10)

df['Rating'].value_counts().sort_index()

Rating
0.5     393068
1.0     776815
1.5     399490
2.0    1640868
2.5    1262797
3.0    4896928
3.5    3177318
4.0    6639798
4.5    2200539
5.0    3612474
Name: count, dtype: int64

In [2]:
movie_stats = df.groupby("Movie_Name").agg(
    average_rating=("Rating", "mean"),
    rating_count=("Rating", "count")
)

movie_stats.head()

movie_stats.sort_values(
    by="rating_count",
    ascending=False
).head(10)


movie_stats.sort_values(
    by="average_rating",
    ascending=False
).head(10)

,average_rating,rating_count
Movie_Name,,
Iran Is Not the Problem (2008),5.0,1
.hack Liminality In the Case of Yuki Aihara,5.0,1
$uperthief: Inside America's Biggest Bank Score (2012),5.0,1
.hack Liminality In the Case of Kyoko Tohno,5.0,1
...E fuori nevica! (2014),5.0,1
The Girl from Flanders (1956),5.0,1
'Master Harold' ... And the Boys (2010),5.0,1
Isabelle Dances Into the Spotlight (2014),5.0,1
"Love, Cheat & Steal (1993)",5.0,1


In [3]:
popular_movies = movie_stats[
    movie_stats["rating_count"] >= 1000
]

popular_movies.sort_values(
    by="average_rating",
    ascending=False
).head(20)

,average_rating,rating_count
Movie_Name,,
Planet Earth II (2016),4.483096,1124
Planet Earth (2006),4.464797,1747
"Shawshank Redemption, The (1994)",4.413576,81482
Band of Brothers (2001),4.398599,1356
"Godfather, The (1972)",4.324336,52498
"Usual Suspects, The (1995)",4.284353,55366
"Godfather: Part II, The (1974)",4.261759,34188
Seven Samurai (Shichinin no samurai) (1954),4.254769,13367
Schindler's List (1993),4.247579,60411


In [4]:
movie_counts = df['Movie_Name'].value_counts()

popular_movies = movie_counts[movie_counts >= 1000].index

filtered_df = df[df['Movie_Name'].isin(popular_movies)]

filtered_df.shape

user_counts = filtered_df['User_Id'].value_counts()

active_users = user_counts[user_counts >= 50].index

filtered_df = filtered_df[
    filtered_df['User_Id'].isin(active_users)
]

filtered_df.shape

filtered_df['Movie_Name'].nunique()

filtered_df['User_Id'].nunique()


99638

In [5]:
from sklearn.preprocessing import LabelEncoder

# Create encoders
user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()

# Encode user IDs
filtered_df["user_index"] = user_encoder.fit_transform(filtered_df["User_Id"])

# Encode movie names
filtered_df["movie_index"] = movie_encoder.fit_transform(filtered_df["Movie_Name"])

filtered_df.head()

print("Users:", filtered_df["user_index"].nunique())
print("Movies:", filtered_df["movie_index"].nunique())

Users: 99638
Movies: 3794


In [6]:
from scipy.sparse import csr_matrix
sparse_matrix = csr_matrix(
    (
        filtered_df["Rating"],
        (
            filtered_df["user_index"],
            filtered_df["movie_index"]
        )
    )
)
print(sparse_matrix.shape)

print(sparse_matrix.nnz)

(99638, 3794)
20198641


In [7]:
from sklearn.neighbors import NearestNeighbors
model = NearestNeighbors(
    metric='cosine',
    algorithm='brute',
    n_neighbors=11
)
model.fit(sparse_matrix.T)
print("Model trained successfully!")

Model trained successfully!


In [8]:
def recommend_movies(movie_name):

    if movie_name not in movie_encoder.classes_:
        print("Movie not found!")
        return

    movie_idx = movie_encoder.transform([movie_name])[0]

    distances, indices = model.kneighbors(
        sparse_matrix.T[movie_idx],
        n_neighbors=11
    )

    print(f"\nRecommendations for: {movie_name}\n")

    for i in range(1, len(indices[0])):
        recommended_movie = movie_encoder.inverse_transform(
            [indices[0][i]]
        )[0]

        similarity = 1 - distances[0][i]

        print(f"{i}. {recommended_movie} (Similarity: {similarity:.2f})")


In [9]:
recommend_movies("Toy Story (1995)")


Recommendations for: Toy Story (1995)

1. Toy Story 2 (1999) (Similarity: 0.63)
2. Forrest Gump (1994) (Similarity: 0.63)
3. Star Wars: Episode IV - A New Hope (1977) (Similarity: 0.62)
4. Jurassic Park (1993) (Similarity: 0.62)
5. Back to the Future (1985) (Similarity: 0.61)
6. Lion King, The (1994) (Similarity: 0.60)
7. Star Wars: Episode VI - Return of the Jedi (1983) (Similarity: 0.60)
8. Aladdin (1992) (Similarity: 0.60)
9. Star Wars: Episode V - The Empire Strikes Back (1980) (Similarity: 0.59)
10. Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981) (Similarity: 0.58)


In [10]:
def get_recommendations(movie_name, n=10):

    if movie_name not in movie_encoder.classes_:
        print("Movie not found!")
        return pd.DataFrame()

    movie_idx = movie_encoder.transform([movie_name])[0]

    distances, indices = model.kneighbors(
        sparse_matrix.T[movie_idx],
        n_neighbors=n + 1
    )

    recommendations = []

    for i in range(1, len(indices[0])):

        recommended_movie = movie_encoder.inverse_transform(
            [indices[0][i]]
        )[0]

        similarity = 1 - distances[0][i]

        movie_info = filtered_df[
            filtered_df["Movie_Name"] == recommended_movie
        ].iloc[0]

        recommendations.append({
            "Movie_Name": recommended_movie,
            "Similarity": similarity,
            "Genre": movie_info["Genre"]
        })

    return pd.DataFrame(recommendations)

In [11]:
recommendations = get_recommendations("Toy Story (1995)")
recommendations

,Movie_Name,Similarity,Genre
0,Toy Story 2 (1999),0.627538,Adventure|Animation|Children|Comedy|Fantasy
1,Forrest Gump (1994),0.625039,Comedy|Drama|Romance|War
2,Star Wars: Episode IV - A New Hope (1977),0.621237,Action|Adventure|Sci-Fi
3,Jurassic Park (1993),0.615696,Action|Adventure|Sci-Fi|Thriller
4,Back to the Future (1985),0.614497,Adventure|Comedy|Sci-Fi
5,"Lion King, The (1994)",0.603698,Adventure|Animation|Children|Drama|Musical|IMAX
6,Star Wars: Episode VI - Return of the Jedi (1983),0.599191,Action|Adventure|Sci-Fi
7,Aladdin (1992),0.598740,Adventure|Animation|Children|Comedy|Musical
8,Star Wars: Episode V - The Empire Strikes Back...,0.590558,Action|Adventure|Sci-Fi
9,Raiders of the Lost Ark (Indiana Jones and the...,0.582244,Action|Adventure


In [12]:
user_id = 1

user_ratings = filtered_df[
    filtered_df["User_Id"] == user_id
].sort_values("Rating", ascending=False)

user_ratings[["Movie_Name", "Rating", "Genre"]].head(20)

,Movie_Name,Rating,Genre


In [13]:
liked_movies = user_ratings[
    user_ratings["Rating"] >= 4.0
]

liked_movies[["Movie_Name", "Rating"]].head(20)

,Movie_Name,Rating


In [14]:
user_ratings[["Movie_Name", "Rating", "Genre"]].head(20)

,Movie_Name,Rating,Genre


In [15]:
liked_movies[["Movie_Name", "Rating"]].head(20)

,Movie_Name,Rating


In [16]:
user_counts = filtered_df["User_Id"].value_counts()

user_counts.head(10)

User_Id
57548     3460
72315     3423
20055     3176
107650    2928
80974     2916
33844     2892
49403     2828
92046     2819
162516    2778
89464     2750
Name: count, dtype: int64

In [17]:
user_id = user_counts.index[0]

print("Selected User:", user_id)
print("Number of ratings:", user_counts.iloc[0])

Selected User: 57548
Number of ratings: 3460


In [18]:
liked_movies = user_ratings[
    user_ratings["Rating"] >= 4.0
]

liked_movies[["Movie_Name", "Rating"]].head(20)

,Movie_Name,Rating


In [19]:
# Check the selected user directly

print("Selected user:", user_id)

print(
    "Rows found:",
    filtered_df[filtered_df["User_Id"] == user_id].shape[0]
)

print(
    "User ID type:",
    type(user_id)
)

print(
    "Column type:",
    filtered_df["User_Id"].dtype
)

Selected user: 57548
Rows found: 3460
User ID type: <class 'numpy.int64'>
Column type: int64


In [20]:
user_ratings = filtered_df.loc[
    filtered_df["User_Id"] == user_id,
    ["Movie_Name", "Rating", "Genre"]
].sort_values(
    "Rating",
    ascending=False
)

print("Number of ratings:", len(user_ratings))

user_ratings.head(20)

Number of ratings: 3460


,Movie_Name,Rating,Genre
8825358,Who Framed Roger Rabbit? (1988),5.0,Adventure|Animation|Children|Comedy|Crime|Fant...
8826712,Kung Fu Hustle (Gong fu) (2004),5.0,Action|Comedy
8824269,Casablanca (1942),5.0,Drama|Romance
8825375,Re-Animator (1985),5.0,Comedy|Horror|Sci-Fi
8825270,Airplane! (1980),5.0,Comedy
8825255,Little Shop of Horrors (1986),5.0,Comedy|Horror|Musical
8825396,Tora! Tora! Tora! (1970),5.0,Action|Drama|War
8824500,Patton (1970),5.0,Drama|War
8826541,Dodgeball: A True Underdog Story (2004),5.0,Comedy
8824463,"Sting, The (1973)",5.0,Comedy|Crime


In [21]:
liked_movies = user_ratings[
    user_ratings["Rating"] >= 4.0
]

print("Movies rated 4 or higher:", len(liked_movies))

liked_movies.head(20)

Movies rated 4 or higher: 226


,Movie_Name,Rating,Genre
8825358,Who Framed Roger Rabbit? (1988),5.0,Adventure|Animation|Children|Comedy|Crime|Fant...
8826712,Kung Fu Hustle (Gong fu) (2004),5.0,Action|Comedy
8824269,Casablanca (1942),5.0,Drama|Romance
8825375,Re-Animator (1985),5.0,Comedy|Horror|Sci-Fi
8825270,Airplane! (1980),5.0,Comedy
8825255,Little Shop of Horrors (1986),5.0,Comedy|Horror|Musical
8825396,Tora! Tora! Tora! (1970),5.0,Action|Drama|War
8824500,Patton (1970),5.0,Drama|War
8826541,Dodgeball: A True Underdog Story (2004),5.0,Comedy
8824463,"Sting, The (1973)",5.0,Comedy|Crime


In [22]:
liked_movies = user_ratings[
    user_ratings["Rating"] >= 4.0
].copy()

print("Liked movies:", len(liked_movies))
liked_movies.head(10)

liked_movie = liked_movies.iloc[0]["Movie_Name"]

print("Liked movie:", liked_movie)
movie_idx = movie_encoder.transform([liked_movie])[0]

print("Movie index:", movie_idx)

Liked movies: 226
Liked movie: Who Framed Roger Rabbit? (1988)
Movie index: 3694


In [23]:
distances, indices = model.kneighbors(
    sparse_matrix.T[movie_idx],
    n_neighbors=6
)

print(indices)
print(distances)

[[3694 1317  347 2167  374  276]]
[[1.12798659e-13 4.10975928e-01 4.53056530e-01 4.91042519e-01
  4.91111786e-01 4.92795328e-01]]


In [24]:
for i in range(1, len(indices[0])):

    movie_index = indices[0][i]

    movie_name = movie_encoder.inverse_transform(
        [movie_index]
    )[0]

    similarity = 1 - distances[0][i]

    print(
        f"{i}. {movie_name} "
        f"(Similarity: {similarity:.3f})"
    )

1. Ghostbusters (a.k.a. Ghost Busters) (1984) (Similarity: 0.589)
2. Beetlejuice (1988) (Similarity: 0.547)
3. Men in Black (a.k.a. MIB) (1997) (Similarity: 0.509)
4. Big (1988) (Similarity: 0.509)
5. Back to the Future (1985) (Similarity: 0.507)


In [25]:
from collections import defaultdict

candidate_scores = defaultdict(float)

In [26]:
for _, row in liked_movies.iterrows():

    liked_movie = row["Movie_Name"]
    user_rating = row["Rating"]

    movie_idx = movie_encoder.transform([liked_movie])[0]

    distances, indices = model.kneighbors(
        sparse_matrix.T[movie_idx],
        n_neighbors=11
    )

    for i in range(1, len(indices[0])):

        recommended_movie_idx = indices[0][i]

        similarity = 1 - distances[0][i]

        recommended_movie = movie_encoder.inverse_transform(
            [recommended_movie_idx]
        )[0]

        score = user_rating * similarity

        candidate_scores[recommended_movie] += score

In [27]:
rated_movies = set(user_ratings["Movie_Name"])

candidate_scores = {
    movie: score
    for movie, score in candidate_scores.items()
    if movie not in rated_movies
}

In [28]:
sorted_recommendations = sorted(
    candidate_scores.items(),
    key=lambda x: x[1],
    reverse=True
)

sorted_recommendations[:10]

[('Terminator 2: Judgment Day (1991)', np.float64(8.431876319045923)),
 ('Minority Report (2002)', np.float64(8.286016237361718)),
 ('Donnie Darko (2001)', np.float64(7.157707652347659)),
 ('Memento (2000)', np.float64(6.517898079934722)),
 ('Untitled Spider-Man Reboot (2017)', np.float64(5.921503738698723)),
 ('Land of the Dead (2005)', np.float64(4.089803066818343)),
 ('Audition (Ôdishon) (1999)', np.float64(3.6113825252019813)),
 ("Child's Play 2 (1990)", np.float64(3.3665241059936606)),
 ('Secret in Their Eyes, The (El secreto de sus ojos) (2009)',
  np.float64(2.6021432931428228)),
 ('Mr. Nobody (2009)', np.float64(1.9362239808617807))]

In [29]:
top_recommendations = sorted_recommendations[:10]

for rank, (movie, score) in enumerate(top_recommendations, start=1):
    print(f"{rank}. {movie}")
    print(f"   Personalized Score: {score:.3f}")
    print()

1. Terminator 2: Judgment Day (1991)
   Personalized Score: 8.432

2. Minority Report (2002)
   Personalized Score: 8.286

3. Donnie Darko (2001)
   Personalized Score: 7.158

4. Memento (2000)
   Personalized Score: 6.518

5. Untitled Spider-Man Reboot (2017)
   Personalized Score: 5.922

6. Land of the Dead (2005)
   Personalized Score: 4.090

7. Audition (Ôdishon) (1999)
   Personalized Score: 3.611

8. Child's Play 2 (1990)
   Personalized Score: 3.367

9. Secret in Their Eyes, The (El secreto de sus ojos) (2009)
   Personalized Score: 2.602

10. Mr. Nobody (2009)
   Personalized Score: 1.936



In [30]:
from collections import defaultdict

weighted_scores = defaultdict(float)
rating_weights = defaultdict(float)

In [31]:
for _, row in liked_movies.iterrows():

    liked_movie = row["Movie_Name"]
    user_rating = row["Rating"]

    movie_idx = movie_encoder.transform([liked_movie])[0]

    distances, indices = model.kneighbors(
        sparse_matrix.T[movie_idx],
        n_neighbors=11
    )

    for i in range(1, len(indices[0])):

        recommended_movie_idx = indices[0][i]

        similarity = 1 - distances[0][i]

        recommended_movie = movie_encoder.inverse_transform(
            [recommended_movie_idx]
        )[0]

        weighted_scores[recommended_movie] += (
            user_rating * similarity
        )

        rating_weights[recommended_movie] += user_rating

In [32]:
rated_movies = set(user_ratings["Movie_Name"])

for movie in rated_movies:
    weighted_scores.pop(movie, None)
    rating_weights.pop(movie, None)

In [33]:
final_scores = {}

for movie in weighted_scores:

    final_scores[movie] = (
        weighted_scores[movie] /
        rating_weights[movie]
    )

In [34]:
sorted_final_recommendations = sorted(
    final_scores.items(),
    key=lambda x: x[1],
    reverse=True
)

In [35]:
for rank, (movie, score) in enumerate(
    sorted_final_recommendations[:10],
    start=1
):

    print(f"{rank}. {movie}")
    print(f"   Personalized Score: {score:.3f}")
    print()

1. Terminator 2: Judgment Day (1991)
   Personalized Score: 0.527

2. Minority Report (2002)
   Personalized Score: 0.473

3. Memento (2000)
   Personalized Score: 0.466

4. Untitled Spider-Man Reboot (2017)
   Personalized Score: 0.456

5. Child's Play 3 (1991)
   Personalized Score: 0.436

6. Donnie Darko (2001)
   Personalized Score: 0.421

7. Child's Play 2 (1990)
   Personalized Score: 0.421

8. Audition (Ôdishon) (1999)
   Personalized Score: 0.380

9. Don't Look Now (1973)
   Personalized Score: 0.371

10. Land of the Dead (2005)
   Personalized Score: 0.341



In [36]:
positive_movies = user_ratings[
    user_ratings["Rating"] >= 4.0
]

negative_movies = user_ratings[
    user_ratings["Rating"] <= 2.5
]

neutral_movies = user_ratings[
    (user_ratings["Rating"] > 2.5) &
    (user_ratings["Rating"] < 4.0)
]

print("Positive:", len(positive_movies))
print("Neutral:", len(neutral_movies))
print("Negative:", len(negative_movies))

Positive: 226
Neutral: 288
Negative: 2946


In [37]:
strong_negative_movies = user_ratings[
    user_ratings["Rating"] <= 1.5
].copy()

print("Strong negative movies:", len(strong_negative_movies))

strong_negative_movies[["Movie_Name", "Rating", "Genre"]].head(20)

Strong negative movies: 2709


,Movie_Name,Rating,Genre
8826769,Aeon Flux (2005),1.5,Action|Sci-Fi
8826470,Troy (2004),1.5,Action|Adventure|Drama|War
8824158,James and the Giant Peach (1996),1.5,Adventure|Animation|Children|Fantasy|Musical
8824870,Lethal Weapon (1987),1.5,Action|Comedy|Crime|Drama
8827371,Shutter Island (2010),1.5,Drama|Mystery|Thriller
8823732,Babe (1995),1.5,Children|Drama
8824806,Fear and Loathing in Las Vegas (1998),1.5,Adventure|Comedy|Drama
8826030,Black Hawk Down (2001),1.5,Action|Drama|War
8826969,Shrek the Third (2007),1.5,Adventure|Animation|Children|Comedy|Fantasy
8826206,"Lord of the Rings: The Two Towers, The (2002)",1.5,Adventure|Fantasy


In [38]:
from collections import defaultdict

positive_scores = defaultdict(float)
negative_scores = defaultdict(float)

In [39]:
for _, row in liked_movies.iterrows():

    liked_movie = row["Movie_Name"]
    user_rating = row["Rating"]

    movie_idx = movie_encoder.transform([liked_movie])[0]

    distances, indices = model.kneighbors(
        sparse_matrix.T[movie_idx],
        n_neighbors=11
    )

    for i in range(1, len(indices[0])):

        recommended_movie_idx = indices[0][i]

        similarity = 1 - distances[0][i]

        recommended_movie = movie_encoder.inverse_transform(
            [recommended_movie_idx]
        )[0]

        positive_scores[recommended_movie] += (
            user_rating * similarity
        )

In [40]:
final_scores_with_negative = {}

all_candidates = set(positive_scores) | set(negative_scores)

for movie in all_candidates:

    positive = positive_scores.get(movie, 0)
    negative = negative_scores.get(movie, 0)

    final_scores_with_negative[movie] = (
        positive - negative
    )

In [41]:
rated_movies = set(user_ratings["Movie_Name"])

for movie in rated_movies:
    final_scores_with_negative.pop(movie, None)

In [42]:
sorted_recommendations_negative = sorted(
    final_scores_with_negative.items(),
    key=lambda x: x[1],
    reverse=True
)

In [43]:
for rank, (movie, score) in enumerate(
    sorted_recommendations_negative[:10],
    start=1
):

    print(f"{rank}. {movie}")
    print(f"   Final Score: {score:.3f}")
    print()

1. Terminator 2: Judgment Day (1991)
   Final Score: 8.432

2. Minority Report (2002)
   Final Score: 8.286

3. Donnie Darko (2001)
   Final Score: 7.158

4. Memento (2000)
   Final Score: 6.518

5. Untitled Spider-Man Reboot (2017)
   Final Score: 5.922

6. Land of the Dead (2005)
   Final Score: 4.090

7. Audition (Ôdishon) (1999)
   Final Score: 3.611

8. Child's Play 2 (1990)
   Final Score: 3.367

9. Secret in Their Eyes, The (El secreto de sus ojos) (2009)
   Final Score: 2.602

10. Mr. Nobody (2009)
   Final Score: 1.936



In [44]:
top_movies = [
    movie for movie, score
    in sorted_final_recommendations[:10]
]

recommendation_df = filtered_df[
    filtered_df["Movie_Name"].isin(top_movies)
][
    ["Movie_Name", "Genre"]
].drop_duplicates()

recommendation_df

,Movie_Name,Genre
93,Terminator 2: Judgment Day (1991),Action|Sci-Fi
209,Minority Report (2002),Action|Crime|Mystery|Sci-Fi|Thriller
376,Memento (2000),Mystery|Thriller
2280,Donnie Darko (2001),Drama|Mystery|Sci-Fi|Thriller
4897,Audition (Ôdishon) (1999),Drama|Horror|Mystery|Romance|Thriller
10965,Child's Play 2 (1990),Horror|Thriller
10966,Child's Play 3 (1991),Comedy|Horror|Thriller
17261,Don't Look Now (1973),Drama|Horror|Thriller
17841,Untitled Spider-Man Reboot (2017),Action|Adventure|Fantasy
30244,Land of the Dead (2005),Action|Horror|Thriller


In [45]:
model_movies = (
    filtered_df[["Movie_Name"]]
    .drop_duplicates()
)

print("Movies in model:", len(model_movies))

Movies in model: 3794


In [46]:
model_movies.to_csv(
    "../data/model_movies.csv",
    index=False
)